Partendo dal codice della lezione, modifica la funzione di preprocessing (map) per iniettare del rumore casuale gaussiano (con media 0 e deviazione standard 0.01) solo durante la fase di normalizzazione. Successivamente, configura la pipeline in modo che il cache() avvenga prima delle shuffle, ma dopo il mapping. Spiega perchè questa sequenza è più efficiente.

In [1]:
import tensorflow as tf
import numpy as np

# --- CREAZIONE DEL'OGGETTO TD.DATASET  ---
#Simuliamo 10.000 campioni con 20 feature ciascuno
X_raw=np.random.uniform(0,255,(10000,20)).astype(np.float32)
y_raw= np.random.randint(0,2,(10000,1)).astype(np.float32)

#Trasformiamo gli array NumPy in un oggetto Dataset
# from_tensor_slices scompone gli array lungo la prima dimensione, creando coppie (X,y) per ogni campione.
dataset=tf.data.Dataset.from_tensor_slices((X_raw,y_raw))

# --- LOGICA DI PIPELINE OTTIMIZZATA ---
#1. MAP: Operazione costosa (CPU)
#Qui facciamo calcolo matematici e generiamo numeri casuali (rumore).
dataset=dataset.map(lambda x,y: (x/255.0 + tf.random.normal(tf.shape(x), mean=0.0, stddev=0.1), y), num_parallel_calls=tf.data.AUTOTUNE )

#2. CACHE: Il 'Salva-risultati'
#POSIZIONE: Dopo il MAP.
#PERCHE': Vogliamo memorizzare il dato già pulito e normalizzato. 
#In questo modo, dalla secondo epoca in poi, il calcolo del 'map' sopra
#viene saltato completamente, risparmiando cicli di CPU preziosi.
dataset=dataset.cache()

#3. SHUFFLE: Il 'Mescolatore'
#POSIZIONE: Dopo il cache.
#PERCHE': Se lo mettessimo PRIMA del cache, l'ordine casuale verrbee memorizzato (congelato).
#Invece, mettendolo DOPO, il cache fornisce i dati (veloce), e lo shuffle li rimescola
#in modo diverso per ogni epoca. Questo garantisce la varietà statistica necessaria
#per evitare che il modello impari la sequenza dei dati invece dei pattern.
dataset=dataset.shuffle(buffer_size=1000)

#4. BATCH e PREFETCH: Gli 'Organizzatori'
#Una volta che i dati sono estratti e mescolati, li raggruppiamo e prepariamo il batch successivo in anticipo (Prefetch).
dataset=dataset.batch(32).prefetch(tf.data.AUTOTUNE)

#ESECUZIONE
model=tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu', input_shape=(20,)),
    tf.keras.layers.Dense(1, activation='sigmoid')
    ])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(dataset, epochs=5)


Epoch 1/5


c:\Users\uberti\.conda\envs\ai_epicode\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.5038 - loss: 0.6974
Epoch 2/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.5125 - loss: 0.6938
Epoch 3/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.5097 - loss: 0.6934
Epoch 4/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.5154 - loss: 0.6923
Epoch 5/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.5217 - loss: 0.6920
